<a href="https://colab.research.google.com/github/Open-Athena/MarinFold/blob/main/notebooks/explore_esm_atlas_distill.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# ESM Atlas (ESMFold2) distilled structures — random sampler & 3D viewer

A tiny **collab notebook for browsing** the
[`open-athena/esm-atlas-esmfold2-distill`](https://huggingface.co/buckets/open-athena/esm-atlas-esmfold2-distill)
bucket — a quality-filtered, novelty-deduplicated slice of the **ESM Atlas** (~66.8M ESMFold2
cluster representatives, materialized to mmCIF) built for MarinFold training-set expansion
([issue #91](https://github.com/Open-Athena/MarinFold/issues/91)).

**What it does:** randomly samples **10 proteins** from the bucket, loads their mmCIFs, and renders
them in an inline [py3Dmol](https://pypi.org/project/py3Dmol/) viewer, cartoon-colored by
**per-residue pLDDT** (the confidence stored in the B-factor column).

- Runs on a **free Colab CPU runtime** — no GPU, no HF login (the bucket is public).
- The bucket is ~1.9 TiB across 3,338 Parquet parts (20k structures each). We never download a whole
  part: we range-read a single **row group** (~500 structures, ~60 MB) from each of a few random parts
  and sample from the pool, so a fresh draw is a few seconds and ~100–200 MB.

> Re-run the sampling cell (or change `SEED`) to draw a new set of 10.


In [ ]:
%pip install -q py3Dmol fsspec aiohttp

In [ ]:
# @title Configuration { run: "auto" }
# How many proteins to display, and how widely to sample.
N_PROTEINS = 10          # @param {type:"integer"}
POOL_PARTS = 3           # @param {type:"integer"}  # random parts to pool from (each contributes ~1 row group / ~60 MB)
SEED       = -1          # @param {type:"integer"}  # -1 = fresh random draw every run; >=0 = reproducible

# --- bucket layout (see the dataset README) -----------------------------------
BUCKET   = "open-athena/esm-atlas-esmfold2-distill"
BASE     = f"https://huggingface.co/buckets/{BUCKET}/resolve"
PART_FMT = BASE + "/structures/parts/part_{:05d}.parquet"
N_PARTS  = 3338          # part_00000 .. part_03337

# Columns we pull from each part (skip the heavy ones we don't need).
COLUMNS = ["entry_id", "sequence", "seq_len", "global_plddt", "ptm", "plddt_std", "cif_content"]


In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import fsspec


def _open_part(part_idx):
    """Open a part for random-access reads over HTTP range requests (no full download)."""
    return pq.ParquetFile(fsspec.open(PART_FMT.format(part_idx)).open())


def sample_proteins(n=N_PROTEINS, pool_parts=POOL_PARTS, seed=SEED):
    """Draw `n` proteins by range-reading one random row group from each of `pool_parts` random parts."""
    rng = np.random.default_rng(None if seed is None or seed < 0 else seed)
    parts = rng.choice(N_PARTS, size=pool_parts, replace=False)
    pool = []
    for part in parts:
        part = int(part)
        pf = _open_part(part)
        rg = int(rng.integers(pf.metadata.num_row_groups))
        tbl = pf.read_row_group(rg, columns=COLUMNS)
        rows = tbl.to_pylist()
        for r in rows:
            r["_part"], r["_row_group"] = part, rg
        pool.extend(rows)
        print(f"  part_{part:05d} row-group {rg}: {len(rows)} structures")
    idx = rng.choice(len(pool), size=min(n, len(pool)), replace=False)
    return [pool[i] for i in idx]


print(f"Sampling {N_PROTEINS} proteins from {POOL_PARTS} random part(s) of {BUCKET} ...")
sampled = sample_proteins()

summary = pd.DataFrame([
    {
        "cell": k,
        "entry_id": p["entry_id"],
        "seq_len": p["seq_len"],
        "pLDDT": round(p["global_plddt"] * 100, 1),   # README: global_plddt is 0-1
        "pTM": round(p["ptm"], 3),
        "plddt_std": round(p["plddt_std"], 3),
        "part": p["_part"],
        "row_group": p["_row_group"],
    }
    for k, p in enumerate(sampled)
])
print(f"\nDrew {len(sampled)} proteins:")
summary


## Molecular viewer

Grid of the sampled structures, cartoon-colored by **per-residue pLDDT** — the ESMFold2 confidence
stored in each atom's B-factor. Grid reads **left-to-right, top-to-bottom**; the `cell` column in the
table above tells you which protein is which. Drag to rotate, scroll to zoom.

**pLDDT scale:** 🔴 low (≤50) → 🟠 → 🟡 → 🟢 → 🔵 high (≥90). Higher = more confident.


In [ ]:
import py3Dmol

# roygb gradient over the B-factor (pLDDT) column: min->red (low conf), max->blue (high conf).
PLDDT_STYLE = {"cartoon": {"colorscheme": {"prop": "b", "gradient": "roygb", "min": 50, "max": 90}}}

NCOL = 5
NROW = -(-len(sampled) // NCOL)  # ceil
view = py3Dmol.view(width=260 * NCOL, height=240 * NROW, viewergrid=(NROW, NCOL), linked=False)
view.setBackgroundColor("0xffffff")

for k, p in enumerate(sampled):
    r, c = divmod(k, NCOL)
    try:
        view.addModel(p["cif_content"], "mmcif", viewer=(r, c))
        view.setStyle({}, PLDDT_STYLE, viewer=(r, c))
        view.zoomTo(viewer=(r, c))
    except Exception as e:  # noqa: BLE001 - one bad structure shouldn't kill the grid
        print(f"[cell {k}] render failed for {p['entry_id']}: {e}")

view.show()


### Zoom in on one protein

Set `IDX` to a `cell` number from the table to inspect a single structure at full size (spinning).


In [ ]:
# @title Single-protein view { run: "auto" }
IDX = 0  # @param {type:"integer"}

p = sampled[IDX]
print(f"{p['entry_id']}   L={p['seq_len']}   "
      f"pLDDT={p['global_plddt'] * 100:.1f}   pTM={p['ptm']:.3f}   "
      f"(part_{p['_part']:05d} / row-group {p['_row_group']})")
print(p["sequence"])

v = py3Dmol.view(width=720, height=520)
v.addModel(p["cif_content"], "mmcif")
v.setStyle({}, PLDDT_STYLE)
v.zoomTo()
v.setBackgroundColor("0xffffff")
v.spin(True)
v.show()


### (Optional) Save the sampled mmCIFs

Writes the drawn structures to `./sampled_cifs/` so you can download them from the Colab file
browser or open them in PyMOL / ChimeraX.


In [ ]:
import pathlib

outdir = pathlib.Path("sampled_cifs")
outdir.mkdir(exist_ok=True)
for p in sampled:
    (outdir / f"{p['entry_id']}.cif").write_text(p["cif_content"])
print(f"Wrote {len(sampled)} mmCIF files to {outdir.resolve()}/")
sorted(x.name for x in outdir.glob("*.cif"))


---

**Dataset:** [`open-athena/esm-atlas-esmfold2-distill`](https://huggingface.co/buckets/open-athena/esm-atlas-esmfold2-distill)
· built for MarinFold [#91](https://github.com/Open-Athena/MarinFold/issues/91).

Derived (quality-filtered, novelty-deduplicated, re-clustered, re-materialized) from the **ESM Atlas
(ESMFold2)**, Biohub — *Language Modeling Materializes a World Model of Protein Biology*, bioRxiv
[2026.06.03.729735](https://doi.org/10.1101/2026.06.03.729735). Both the source Atlas and this derived
dataset are released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
